# 1 - Config & Import

## 1.1 - Config

In [1]:
# 🔧 config import
import os
from core.config.logger_config import colored_logger
logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-29 17:15:54] INFO - 726159153.py - Logger initialized (Notebook)


## 1.2 - Import

In [2]:
# 📦 External libs
import plotly.graph_objects as go
import importlib
import pandas as pd

# 📂 Internal modules
import core.data.fetcher as fetcher
import core.data.features as features
import core.data.labels as labels

symbol = 'BTCUSDT'
start_date = '2025-06-25'
end_date = "2025-06-27"
interval = '1min'

def reload_modules():
    importlib.reload(features)
    importlib.reload(labels)
    importlib.reload(fetcher)

    Fetcher = fetcher.DataFetcher(symbol, start_date, end_date, interval)
    Fetcher.load_from_csv_dom_bybit(directory=os.getcwd())
    Features = features.Features(Fetcher.raw_data)
    return Features

Features = reload_modules()
print(Features.data.head())

[2025-06-29 17:15:58] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-06-29 17:15:58] INFO - features.py - Logger initialized (features.py)
[2025-06-29 17:15:58] INFO - labels.py - Logger initialized (labels.py)
[2025-06-29 17:15:58] INFO - features.py - Logger initialized (features.py)
[2025-06-29 17:15:58] INFO - labels.py - Logger initialized (labels.py)
[2025-06-29 17:15:58] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-06-29 17:15:58] INFO - fetcher.py - 📥 Data loaded from: C:\Users\valer\Desktop\Trading\Trading_Bot\Project\pipeline/BTCUSDT_2025-06-25_2025-06-27_1min_dom.csv


                 time      open      high       low     close  ask_size  \
0 2025-06-25 00:00:00  106087.9  106130.0  106072.0  106072.1  3.270138   
1 2025-06-25 00:01:00  106072.0  106097.2  106063.0  106071.8  1.049900   
2 2025-06-25 00:02:00  106071.9  106071.9  106022.8  106022.9  2.298718   
3 2025-06-25 00:03:00  106022.8  106079.7  106015.3  106065.8  1.608724   
4 2025-06-25 00:04:00  106065.8  106083.0  106041.0  106041.0  1.684092   

   bid_size  
0  4.575117  
1  0.560414  
2  0.002983  
3  2.392039  
4  1.465795  


# 2 - Features

## Variables

In [3]:
# Resample with vwap
resample_period = '5min'

# Pivot Points
pivot_left = 15
pivot_right = 15

# Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

# Std
std_window = 10

# Mean
mean_window = 10

## Process

In [4]:
Features = reload_modules()
Features.data["volume"] = Features.data["bid_size"] + Features.data["ask_size"]
print(Features.data.shape)
print(Features.data.columns)
print(Features.data.head())

# Step 1: Resample with VWAP (Liquidity)
Features.resample_with_vwap(resample_period)

# Step 2: Market Sessions (Market Context)
Features.market_sessions()

# Step 3: Pivot Points (Support/Resistance)
Features.Pivot_Points(pivot_left, pivot_right)

# Step 4: Volume Pivot Points (Volume-based S/R)
Features.volume_Pivot_Points(duration_min, n_cross, std_factor)

# Step 5: Volume Delta (Momentum)
Features.add_volume_delta()

# Step 6: Cumulative Volume Delta - CVD (Momentum)
Features.add_cvd()

# Step 7: Order Book Imbalance - OBI (Pressure)
Features.add_obi()

# Step 8: Price Change (Price Action)
Features.add_price_change()

# Step 9: Reaction Ratio (Liquidity)
Features.add_reaction_ratio()

# Step 10: Rolling Standard Deviation of Price (Volatility)
Features.add_rolling_std_price(std_window)

# Step 11: Rolling Mean of CVD (Smoothed Flow)
Features.add_rolling_mean_cvd(mean_window)

Features.data["volume"] = Features.data["bid_size"] + Features.data["ask_size"]
print(Features.data.shape)
print(Features.data.columns)
print(Features.data.head())


[2025-06-29 17:16:13] INFO - features.py - Logger initialized (features.py)
[2025-06-29 17:16:13] INFO - labels.py - Logger initialized (labels.py)
[2025-06-29 17:16:13] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-06-29 17:16:13] INFO - fetcher.py - 📥 Data loaded from: C:\Users\valer\Desktop\Trading\Trading_Bot\Project\pipeline/BTCUSDT_2025-06-25_2025-06-27_1min_dom.csv
[2025-06-29 17:16:13] INFO - features.py - Starting resampling with period: 5min
[2025-06-29 17:16:13] INFO - features.py - Basic OHLCV resampling done.
[2025-06-29 17:16:13] INFO - features.py - VWAP calculation completed.
[2025-06-29 17:16:13] INFO - features.py - Resampling successful. Resulting shape: (864, 8)
[2025-06-29 17:16:13] INFO - features.py - Starting market session flag generation.
[2025-06-29 17:16:13] INFO - features.py - Datetime index successfully converted and localized.
[2025-06-29 17:16:13] INFO - features.py - Market session flags successfully added.
[2025-06-29 17:16:13] INFO - feat

(4320, 8)
Index(['time', 'open', 'high', 'low', 'close', 'ask_size', 'bid_size',
       'volume'],
      dtype='object')
                 time      open      high       low     close  ask_size  \
0 2025-06-25 00:00:00  106087.9  106130.0  106072.0  106072.1  3.270138   
1 2025-06-25 00:01:00  106072.0  106097.2  106063.0  106071.8  1.049900   
2 2025-06-25 00:02:00  106071.9  106071.9  106022.8  106022.9  2.298718   
3 2025-06-25 00:03:00  106022.8  106079.7  106015.3  106065.8  1.608724   
4 2025-06-25 00:04:00  106065.8  106083.0  106041.0  106041.0  1.684092   

   bid_size    volume  
0  4.575117  7.845255  
1  0.560414  1.610314  
2  0.002983  2.301701  
3  2.392039  4.000763  
4  1.465795  3.149887  


[2025-06-29 17:16:14] INFO - features.py - Initial pivot marking complete.
[2025-06-29 17:16:15] INFO - features.py - Pivot filtering logic applied (highs/lows lists, broken levels).
[2025-06-29 17:16:15] INFO - features.py - Pivot point detection completed successfully.
[2025-06-29 17:16:15] INFO - features.py - Starting volume_Pivot_Points with duration_min=240, n_cross=7, std_factor=1.0
[2025-06-29 17:16:15] INFO - features.py - VWAP and speed computed.
[2025-06-29 17:16:15] INFO - features.py - Detected 25 upward and 26 downward VWAP crosses.
[2025-06-29 17:16:16] INFO - features.py - Calculated rolling pivot boundaries and last cross prices.
[2025-06-29 17:16:16] INFO - features.py - volume pivot point detection completed successfully.
[2025-06-29 17:16:16] INFO - features.py - Adding 'volume_delta' column...
[2025-06-29 17:16:16] INFO - features.py - 'volume_delta' column added.
[2025-06-29 17:16:16] INFO - features.py - Adding 'CVD' column...
[2025-06-29 17:16:16] INFO - feature

(864, 27)
Index(['open', 'high', 'low', 'close', 'volume', 'bid_size', 'ask_size',
       'VWAP_5m', 'London_open', 'NY_open', 'HK_open', 'Dif_low_Pivot',
       'Dif_high_Pivot', 'low_Pivot', 'high_Pivot', 'Rolling_VWAP_240min',
       'volume_low_Pivot', 'volume_high_Pivot', 'Dif_volume_low_Pivot',
       'Dif_volume_high_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd'],
      dtype='object')
                         open      high       low     close     volume  \
time                                                                     
2025-06-25 00:00:00  106087.9  106130.0  106015.3  106041.0  18.907920   
2025-06-25 00:05:00  106041.0  106218.3  106034.1  106218.3  27.845968   
2025-06-25 00:10:00  106218.3  106454.5  106149.1  106440.7  60.408618   
2025-06-25 00:15:00  106440.7  106441.7  106160.2  106267.9  40.931766   
2025-06-25 00:20:00  106267.9  106268.0  106101.2  106132.3  22.556447   

            

In [ ]:
import pandas as pd

# Set option to display all columns
pd.set_option('display.max_columns', None)
# Display the DataFrame
pd.set_option('display.max_rows', None)
display(Features.data)

## Visualization

In [10]:
from lightweight_charts import Chart
import pandas as pd

df = pd.read_csv("BTCUSDT_23-06-2025_27-06-2025_1m_ohlcv.csv")
print(df)

# Close any existing chart
try:
    chart.exit()
except NameError:
    pass

# Create chart with title including symbol and interval, and launch maximized
chart = Chart(
    title=f"{symbol} – {interval}",
    toolbox=True,
    maximize=True
)
# Plot candlestick + volume
chart.set(df[['time', 'open', 'high', 'low', 'close', 'volume']])
#----------------------------------------------------------------------
# Overlay Rolling VWAP indicator
rvwap_line = chart.create_line(
    name="Rolling_VWAP_240min",  # must match DataFrame column name
    color="#FF9900",
    style="solid",
    width=2,
    price_line=False,
    price_label=True
).set(df[['time', 'Rolling_VWAP_240min']].dropna())
#----------------------------------------------------------------------
def on_toggle(chart_obj):
    btn = chart_obj.topbar['vwap_toggle']
    if btn.value == "on":
        rvwap_line.show_data()
    else:
        rvwap_line.hide_data()
    chart_obj.fit()

# Add toggle button (toggle=True enables state)
chart.topbar.button(
    name="vwap_toggle",
    button_text="VWAP",
    toggle=True,
    separator=False,
    align="right",
    func=on_toggle
)
#----------------------------------------------------------------------
# Display chart without blocking
chart.show(block=False)

                     time      open      high       low     close    volume
0     2025-06-22 22:00:00   99136.0   99547.4   99084.8   99492.7   554.595
1     2025-06-22 22:01:00   99492.7   99647.2   99486.3   99591.6   696.955
2     2025-06-22 22:02:00   99588.7   99655.4   99526.2   99529.9   440.510
3     2025-06-22 22:03:00   99529.9   99694.1   99502.1   99694.1   363.372
4     2025-06-22 22:04:00   99694.0   99813.1   99650.0   99781.1   641.497
5     2025-06-22 22:05:00   99781.0  100114.3   99771.5   99996.0  1456.311
6     2025-06-22 22:06:00   99996.1  100050.0   99894.0   99941.7   852.862
7     2025-06-22 22:07:00   99941.8  100167.9   99941.7  100134.1   601.751
8     2025-06-22 22:08:00  100134.2  100250.0  100124.3  100227.6   841.788
9     2025-06-22 22:09:00  100227.6  100307.5  100122.5  100260.6   646.474
10    2025-06-22 22:10:00  100260.6  100312.0  100222.0  100276.1   363.598
11    2025-06-22 22:11:00  100276.1  100454.3  100276.0  100454.1   588.237
12    2025-0

KeyError: "['Rolling_VWAP_240min'] not in index"

In [7]:
import asyncio
from lightweight_charts import Chart
import pandas as pd

df = Features.data.reset_index()

# Close any previous chart
try:
    chart.exit()
except NameError:
    pass

# Create chart
chart = Chart(title=f"{symbol} – {interval}", toolbox=True, maximize=True)
chart.set(df[['time','open','high','low','close','volume']])

# Add Rolling VWAP and start hidden
rvwap = df[['time','Rolling_VWAP_240min']].dropna()
rvwap_line = chart.create_line(
    name="Rolling_VWAP_240min",
    color="#FF9900", style="solid", width=2,
    price_line=False, price_label=True
)
rvwap_line.set(rvwap)

def on_toggle(chart_obj):
    btn = chart_obj.topbar['vwap_toggle']
    if btn.value == "on":
        rvwap_line.show_data()
    else:
        rvwap_line.hide_data()
    chart_obj.fit()

# Add toggle button (toggle=True enables state)
chart.topbar.button(
    name="vwap_toggle",
    button_text="VWAP",
    toggle=True,
    separator=False,
    align="right",
    func=on_toggle
)

# Display chart asynchronously in Jupyter
async def display():
    await chart.show_async()

asyncio.get_event_loop().create_task(display())

<Task pending name='Task-5' coro=<display() running at C:\Users\valer\AppData\Local\Temp\ipykernel_21140\430690977.py:49>>

# 3 - Label Data

## Variables

In [ ]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

## Process

In [ ]:
reload_modules()

# Step 0: Initialize Labels Dataframe
Labels_Data = Class.Data.Labels.Labels(Features_Data.data)

# Step 1: Categorize Pivot Points
Labels_Data.categorize_pivot_points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.categorize_colume_pivot_points(look_forward)

## Visualization

In [ ]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

df = Labels_Data.data.copy()[-300:]

# Fréquences des 0 et 1 pour les colonnes cibles
frequency = df[["Low_Below_Pivot", "High_Above_Pivot"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['High_Above_Pivot'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['Low_Below_Pivot'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Low_Pivot'
))

# Trace High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

# Fréquences des 0 et 1 pour les colonnes VWAP
frequency = df[["VWAP_Below_Volume_Low", "VWAP_Above_Volume_High"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['VWAP_Above_Volume_High'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['VWAP_Below_Volume_Low'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Volume_Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Volume_Low_Pivot'
))

# Trace Volume_High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='Volume_High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Volume Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()

#-------------------------------------------------------------------------------